# 07a — Retrieval from first principles — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/07.md) · [Course map](../PLAN.md) · [Project](../../projects/stage07/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Chunk documents with stable provenance
- Rank keyword matches
- Measure retrieval separately from answers


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

Retrieval selects evidence before an answer is generated. Parsing converts PDF, Markdown, HTML, DOCX, JSON, or database records into text and metadata. Preserve document ID, page or section, source version, tenant, and permissions. OCR and tables need special care; a parser can silently scramble numbers.

Chunking balances context completeness against retrieval precision. Fixed-size chunks are simple; paragraph and semantic chunks follow content boundaries. Overlap can preserve context but also duplicates evidence and consumes space. Use the model tokenizer for actual token limits; the exercise uses whitespace words as a transparent baseline.

Keyword methods match literal terms. BM25 adjusts term frequency using document length and rarity. Dense embeddings can retrieve paraphrases but may miss exact identifiers. Hybrid retrieval combines complementary ranked lists; reranking scores query-candidate pairs with a more expensive model. First establish a measured lexical baseline so you can tell whether embeddings add value.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import re
print(re.findall(r"[a-z0-9]+", "Revenue grew 12% in 2025.".lower()))


## Exercise 1: Overlapping chunks

Return word chunks with size>0 and 0<=overlap<size. Avoid an extra final chunk made only of overlap.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def chunks(text, size=4, overlap=1):
    raise NotImplementedError


In [ ]:
assert chunks("a b c d e f", 4, 1) == ["a b c d", "d e f"]
assert chunks("") == []
expect_error(ValueError, lambda: chunks("a", 2, 2))
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Stop when the current chunk reaches the final word.

</details>


## Exercise 2: Lexical retrieval

For dicts with id,text, rank documents by number of shared unique lowercase alphanumeric terms; omit zero scores, break ties by id.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def retrieve(query, documents, k=3):
    raise NotImplementedError


In [ ]:
docs = [{"id": "a", "text": "Revenue margin"}, {"id": "b", "text": "Password reset"}]
assert retrieve("revenue", docs) == ["a"]
assert retrieve("unicorn", docs) == []
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Score first; sort with explicit tie-breaking.

</details>


## Exercise 3: Reciprocal rank fusion

Combine ranked ID lists using sum(1/(c+rank)), rank starting at 1. Count an ID at most once per list; tie-break by ID.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def rrf(rankings, c=60):
    raise NotImplementedError


In [ ]:
assert rrf([["a", "b"], ["b", "c"]])[0] == "b"
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Fuse ranks when raw score scales differ.

</details>


## Independent transfer

Build a private document index with chunk IDs, source offsets, parser version, and content hashes. Compare keyword, embedding, hybrid, and reranked recall on the same labeled questions.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why keep source offsets?

2. What can reranking fix?


## Reading and review

- [Primary reference 1](https://github.com/pgvector/pgvector)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
